In [76]:
from collections import defaultdict
from collections import Counter
from tqdm import tqdm
import pandas as pd
import numpy as np
import os

In [2]:
root = "/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation"

In [3]:
def get_splits(path):
    splits = os.listdir(path)
    return [i.replace(".csv", "") for i in sorted(splits)]

def get_pockets():
    df = pd.read_csv("/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation/processed/pocket_detection_data.csv")
    pockets = [f"{i.replace('.pdb', '')}_pocket_{j}" for i, j in zip(df['File name'], df['Pocket number'])]
    return sorted(pockets)

In [4]:
# Get splits (994)
SPLITS = get_splits(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", 'A_proteins'))

# Get pockets (276)
POCKETS = get_pockets()

In [5]:
A_pockets = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "A_pockets.csv"))
B_pockets = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "B_pockets.csv"))
A_proteins = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "A_proteins.csv"))
B_proteins = pd.read_csv(os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "B_proteins.csv"))

# Check that all pockets have same number of cpds associated
assert set(Counter(B_pockets['pocket']).values()) == set([1000])
assert set(Counter(B_proteins['pocket']).values()) == set([13000])

# Drop pocket column
B_pockets = B_pockets.drop(columns=['pocket'])
B_proteins = B_proteins.drop(columns=['pocket'])

# Include label
A_pockets['label'] = "A_pockets" 
B_pockets['label'] = "B_pockets" 
A_proteins['label'] = "A_proteins" 
B_proteins['label'] = "B_proteins"

# Merge all, sort and drop duplicates
COMPOUNDS = pd.concat([A_pockets, B_pockets, A_proteins, B_proteins]).sort_values(by=['split', 'index']).drop_duplicates(subset=['split', 'index']).reset_index(drop=True)

In [32]:
print(len(A_pockets) + len(B_pockets) + len(A_proteins) + len(B_proteins))
print(len(COMPOUNDS))
print((len(A_pockets) + len(B_pockets) + len(A_proteins) + len(B_proteins)) - len(COMPOUNDS))

1049000
1047749
1251


In [33]:
PATH_TO_SPLITS = os.path.join(root, "tmp")
PATH_TO_OUTPUT = os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "selected_compounds")
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

In [107]:
ANNOTATED_COMPOUNDS = []

for split in SPLITS[:50]:

    # Identify subset
    df = COMPOUNDS[COMPOUNDS['split'] == split].reset_index(drop=True)
    inds = df['index'].to_numpy()

    # Load split info
    smiles_ids = pd.read_csv(os.path.join(PATH_TO_SPLITS, f"{split}_SMILES_IDs.tsv.zip"), sep='\t')
    smiles_ids = smiles_ids.iloc[inds].reset_index()
    assert (smiles_ids['index'] == df['index']).all

    # Concatanate
    df = pd.concat([df, smiles_ids], axis=1)

    # Save
    df.to_csv(os.path.join(PATH_TO_OUTPUT, f"{split}.csv"), index=False)

In [108]:
def get_splits(path):
    splits = os.listdir(path)
    return [i.replace(".csv", "") for i in sorted(splits)]

def get_pockets():
    df = pd.read_csv("/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation/processed/pocket_detection_data.csv")
    pockets = [f"{i.replace('.pdb', '')}_pocket_{j}" for i, j in zip(df['File name'], df['Pocket number'])]
    return sorted(pockets)


In [109]:
PATH_TO_SELECTED_COMPOUNDS = os.path.join(root, "processed", "unidock_REAL_docking", "inference_10B", "selected_compounds")

In [110]:
df = []
for split in SPLITS:
    try:
        df_ = pd.read_csv(os.path.join(os.path.join(PATH_TO_OUTPUT, f"{split}.csv")))
        df.append(df_)
    except:
        pass
df = pd.concat(df, ignore_index=True)
rng = np.random.default_rng(42)
df["rd"] = rng.random(len(df))

CUSTOM_ORDER = ['A_proteins', 'A_pockets', 'B_proteins', 'B_pockets']
df["label"] = pd.Categorical(df["label"], categories=CUSTOM_ORDER, ordered=True)
df = df.sort_values(["label", "rd"], ascending=[True, True], kind="stable")

In [137]:
MAX_SYNTHON = 10
SYNTHON_COUNTS = defaultdict(int)
KEEP = []

for id_ in tqdm(df['id']):

    if id_.startswith("m_") == False and id_.startswith("s_") == False:
        raise TypeError("ID not starting with m nor s. Please revise")
    
    
    synthons = id_.replace("m_", "").replace("s_", "").split("____")
    keep = True
    tmp_dict = defaultdict(int)

    # tmp dict with synthons
    for synthon in synthons:
        tmp_dict[synthon] += 1

    # check valid compound
    for synthon in synthons:
        if SYNTHON_COUNTS[synthon] + tmp_dict[synthon] <= MAX_SYNTHON:
            pass
        else:
            keep = False
            break

    # add compound if valid
    if keep == True:
        KEEP.append(True)
        for synthon in synthons:
            SYNTHON_COUNTS[synthon] += 1
        
    else:
        KEEP.append(False)

df['keep'] = KEEP
d = Counter(KEEP)
print(round(100 * d[True] / (d[True] + d[False]), 2), "%")

100%|██████████| 26456/26456 [00:00<00:00, 947548.03it/s]

3.0 %
